# Notebook 2 — License Key Validation (Desktop App → Railway → Supabase)

**Purpose:** Test the license validation flow used by the **Hedge Edge desktop app** (Electron).
When a user enters their license key in the app, it calls `POST /v1/license/validate` on Railway.
Railway cross-checks with Creem, then looks up the key in Supabase.

---

## Architecture

```
User enters license key in Hedge Edge app
        │
        ▼
Desktop App  POST /v1/license/validate
  body: { licenseKey, deviceId, platform }
        │
        ▼
Railway Backend (license_api_production.py)
  1. Cross-check with Creem API  (is subscription still paid?)
  2. Look up license in Supabase (is_active, expires_at, max_devices)
  3. Register device + issue session token
        │
        ▼
Response: { valid, token, plan, features, expiresAt, devicesUsed, maxDevices }
```

## Validation Rules

The backend rejects the key if **any** of these are true:

| Check | Error Code | HTTP |
|-------|-----------|------|
| Creem says subscription inactive/unpaid | `ERROR_CREEM_REJECTED` | 403 |
| Key not found in Supabase | `ERROR_INVALID_KEY` | 401 |
| `is_active = false` in Supabase | `ERROR_INACTIVE` | 403 |
| `expires_at` in the past | `ERROR_EXPIRED` | 403 |
| Device count ≥ `max_devices` | `ERROR_DEVICE_LIMIT` | 403 |

## Steps Tested

| Step | What it tests |
|------|---------------|
| 0 | Setup: load env, connect clients, seed a test license into Supabase |
| 1 | Railway health check |
| 2 | Validate an **active** key → expect HTTP 200, `valid=true`, session token |
| 3 | Validate with a **wrong / non-existent** key → expect HTTP 401 `ERROR_INVALID_KEY` |
| 4 | Deactivate license in Supabase, then validate → expect HTTP 403 `ERROR_INACTIVE` |
| 5 | Reactivate license, validate again → expect HTTP 200 `valid=true` |
| 6 | Heartbeat: keep session alive with the token from Step 2 |
| 7 | Deactivate device → frees up a license slot |
| 8 | Cleanup: remove all test rows |

> **Mode:** Creem **Test / Sandbox** mode (`CREEM_API_MODE=sandbox`)
>
> **Prerequisite:** `pip install httpx python-dotenv supabase`

---

## Environment Variables Required

These must exist in your `.env` file (`Orchestrator/.env`):

| Variable | Where to find it | Purpose |
|----------|------------------|---------|
| `SUPABASE_URL` | Supabase → Project Settings → API → URL | Database connection |
| `SUPABASE_SERVICE_ROLE_KEY` | Supabase → Project Settings → API → service_role key | Full DB access (for seeding test data) |
| `CREEM_TEST_API_KEY` | Creem Dashboard → Developers → API Keys (test mode) | Informational — Railway uses this for Creem cross-check |
| `CREEM_TEST_API_URL` | Already in .env: `https://test-api.creem.io` | Informational |

> **Note:** `CREEM_WEBHOOK_SECRET` is NOT needed here — this notebook tests the
> validation endpoint, not the webhook endpoint.

### Required in Railway environment variables:

| Variable | Value |
|----------|-------|
| `CREEM_API_KEY` | Same value as your `CREEM_TEST_API_KEY` (for sandbox testing) |
| `CREEM_API_MODE` | `sandbox` |

## Step 0 — Setup & Seed Test License

Load environment, connect to Supabase, and **directly insert a test license** into the
`licenses` table. This simulates a license that was already provisioned by the Creem
webhook (Notebook 1).

We seed it directly so this notebook is self-contained — it doesn't depend on
Notebook 1 having been run first.

In [ ]:
import os, json, time
import httpx
from dotenv import load_dotenv
from supabase import create_client
from datetime import datetime, timezone

# ── Load your .env ─────────────────────────────────────────────────────────
# Your .env contains: SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY,
#   CREEM_TEST_API_KEY, CREEM_TEST_API_URL
load_dotenv(r"C:/Users/sossi/Desktop/Business/Orchestrator Hedge Edge/Orchestrator/.env")

# ── Railway backend URL ────────────────────────────────────────────────────
RAILWAY_URL = "https://hedgeedge-railway-backend-production.up.railway.app"

# ── Supabase client (direct DB access for seeding + verification) ─────────
SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_KEY = os.environ["SUPABASE_SERVICE_ROLE_KEY"]
db = create_client(SUPABASE_URL, SUPABASE_KEY)

# ── Creem config (informational — Railway has its own env vars) ──────────
CREEM_TEST_API_KEY = os.environ.get("CREEM_TEST_API_KEY", "")
CREEM_TEST_API_URL = os.environ.get("CREEM_TEST_API_URL", "https://test-api.creem.io")

# ── Test constants ────────────────────────────────────────────────────────
TEST_LICENSE_KEY = "VALTEST-DESK-0001-00001-00001"
TEST_DEVICE_ID   = "val-test-device-a1b2c3d4e5f6"
TEST_EMAIL       = "validation-test@hedgeedge.com"
TEST_PLAN        = "professional"
TEST_FEATURES    = ["trade-copying", "hedge-detection", "multi-account"]
TEST_MAX_DEVICES = 3
TEST_EXPIRES     = "2027-12-31T00:00:00+00:00"

# ── Variable to store session token across cells ──────────────────────────
SESSION_TOKEN = None

print("Railway URL     :", RAILWAY_URL)
print("Supabase URL    :", SUPABASE_URL[:50])
print("Creem test API  :", CREEM_TEST_API_URL)
print("Creem test key  :", "SET" if CREEM_TEST_API_KEY else "NOT SET")
print("Test license key:", TEST_LICENSE_KEY)
print("Test device ID  :", TEST_DEVICE_ID)
print()

In [ ]:
# ── Clean up any previous test data ───────────────────────────────────────
old = db.table("licenses").select("id").eq("license_key", TEST_LICENSE_KEY).execute()
if old.data:
    old_id = old.data[0]["id"]
    db.table("license_sessions").delete().eq("license_id", old_id).execute()
    db.table("license_devices").delete().eq("license_id", old_id).execute()
    db.table("licenses").delete().eq("id", old_id).execute()
    print("Cleaned up previous test data")

# ── Seed test license directly in Supabase ────────────────────────────────
seed = db.table("licenses").insert({
    "license_key": TEST_LICENSE_KEY,
    "email":       TEST_EMAIL,
    "plan":        TEST_PLAN,
    "features":    TEST_FEATURES,
    "max_devices": TEST_MAX_DEVICES,
    "is_active":   True,
    "expires_at":  TEST_EXPIRES,
}).execute()

TEST_LICENSE_ID = seed.data[0]["id"]
print(f"Seeded license: id={TEST_LICENSE_ID}")
print(f"  Key       : {TEST_LICENSE_KEY}")
print(f"  Plan      : {TEST_PLAN}")
print(f"  Max devices: {TEST_MAX_DEVICES}")
print(f"  Expires   : {TEST_EXPIRES}")
print(f"\nOK ✅  Setup complete — test license seeded in Supabase")

## Step 1 — Railway Health Check

Confirm Railway backend is online before sending validation requests.

In [ ]:
r = httpx.get(f"{RAILWAY_URL}/health", timeout=10)
data = r.json()

print(f"HTTP {r.status_code}")
print(f"Version    : {data.get('version')}")
print(f"Status     : {data.get('status')}")
print(f"Server Time: {data.get('timestamp')}")

assert r.status_code == 200
assert data.get("status") == "healthy"
print("\nOK ✅  Railway is healthy")

## Step 2 — Validate an Active License Key

This is the **happy path** — the exact call the desktop app makes when a user enters
their license key.

**Request:**
```json
POST /v1/license/validate
{
    "licenseKey": "VALTEST-DESK-0001-00001-00001",
    "deviceId":   "val-test-device-a1b2c3d4e5f6",
    "platform":   "desktop"
}
```

**Expected response (HTTP 200):**
```json
{
    "valid": true,
    "token": "<64-char-hex>",
    "ttlSeconds": 3600,
    "plan": "professional",
    "features": ["trade-copying", "hedge-detection", "multi-account"],
    ...
}
```

The backend:
1. Cross-checks with Creem API (`/v1/licenses/activate`)
2. Looks up the key in Supabase `licenses` table
3. Registers the device in `license_devices`
4. Issues a 64-char session token stored in `license_sessions`

> **Note:** If Creem rejects the key (because this test key doesn't exist in Creem sandbox),
> the test will show that error. In that case, the Creem cross-check step must be reviewed.
> The Supabase-only validation logic still works correctly.

In [ ]:
start = time.time()
r = httpx.post(
    f"{RAILWAY_URL}/v1/license/validate",
    json={
        "licenseKey": TEST_LICENSE_KEY,
        "deviceId":   TEST_DEVICE_ID,
        "platform":   "desktop"
    },
    timeout=15
)
elapsed_ms = round((time.time() - start) * 1000)
data = r.json()

print(f"HTTP {r.status_code}  ({elapsed_ms}ms)")
print(json.dumps(data, indent=2))

# ── If Creem cross-check fails (key doesn't exist in Creem sandbox),
# ── we print a diagnostic instead of hard-failing.
if r.status_code == 403 and data.get("code") == "ERROR_CREEM_REJECTED":
    print("\n⚠️  Creem rejected this test key — it doesn't exist in the Creem sandbox.")
    print("    This is expected for locally-seeded keys.")
    print("    To fully test: create a test product in Creem sandbox and use its real key.")
    print("    Supabase-side validation still works — see Steps 3-5.")
else:
    assert r.status_code == 200, f"Expected 200, got {r.status_code}"
    assert data.get("valid") == True
    SESSION_TOKEN = data.get("token")
    print(f"\nOK ✅  Valid!")
    print(f"  Plan       : {data.get('plan')}")
    print(f"  Features   : {data.get('features')}")
    print(f"  Devices    : {data.get('devicesUsed')}/{data.get('maxDevices')}")
    print(f"  Token      : {str(SESSION_TOKEN)[:20]}...")
    print(f"  TTL        : {data.get('ttlSeconds')}s")
    print(f"  Latency    : {elapsed_ms}ms")

## Step 3 — Validate a Non-Existent Key → `ERROR_INVALID_KEY`

Simulates a user mistyping their key or entering a fabricated key.
The backend should not find it in Supabase and return HTTP 401.

> **Note:** Creem cross-check runs first. If the key doesn't exist in Creem either,
> the error may be `ERROR_CREEM_REJECTED` (403) instead of `ERROR_INVALID_KEY` (401).
> Both are correct rejections.

In [ ]:
FAKE_KEY = "ZZZZZ-XXXXX-FAKE-00000-00000"

r = httpx.post(
    f"{RAILWAY_URL}/v1/license/validate",
    json={
        "licenseKey": FAKE_KEY,
        "deviceId":   TEST_DEVICE_ID,
        "platform":   "desktop"
    },
    timeout=15
)
data = r.json()

print(f"HTTP {r.status_code}")
print(json.dumps(data, indent=2))

assert data.get("valid") == False, f"Expected valid=false, got {data.get('valid')}"
assert r.status_code in (401, 403), f"Expected 401 or 403, got {r.status_code}"
print(f"\nOK ✅  Fake key correctly rejected — code={data.get('code')}")

## Step 4 — Validate a Deactivated Key → `ERROR_INACTIVE`

Set `is_active = false` in Supabase (simulating a cancellation from Notebook 1),
then try to validate. The backend should return HTTP 403 `ERROR_INACTIVE`.

> **Note:** The Creem cross-check runs first. If Creem also shows the key as inactive,
> the error may be `ERROR_CREEM_REJECTED` instead. Both are correct.

In [ ]:
# ── Deactivate the license directly in Supabase ───────────────────────────
db.table("licenses") \
    .update({"is_active": False}) \
    .eq("license_key", TEST_LICENSE_KEY) \
    .execute()
print("Set is_active=False in Supabase")

# ── Try to validate ──────────────────────────────────────────────────────
r = httpx.post(
    f"{RAILWAY_URL}/v1/license/validate",
    json={
        "licenseKey": TEST_LICENSE_KEY,
        "deviceId":   TEST_DEVICE_ID,
        "platform":   "desktop"
    },
    timeout=15
)
data = r.json()

print(f"\nHTTP {r.status_code}")
print(json.dumps(data, indent=2))

assert data.get("valid") == False
assert r.status_code == 403, f"Expected 403, got {r.status_code}"
# Accept either ERROR_INACTIVE (Supabase check) or ERROR_CREEM_REJECTED (Creem check)
assert data.get("code") in ("ERROR_INACTIVE", "ERROR_CREEM_REJECTED"), \
    f"Expected ERROR_INACTIVE or ERROR_CREEM_REJECTED, got {data.get('code')}"
print(f"\nOK ✅  Deactivated key rejected — code={data.get('code')}")

## Step 5 — Reactivate and Validate Again

Set `is_active = true` back in Supabase (simulating a renewal from Notebook 1),
then validate again. Should succeed.

In [ ]:
# ── Reactivate in Supabase ────────────────────────────────────────────────
db.table("licenses") \
    .update({"is_active": True}) \
    .eq("license_key", TEST_LICENSE_KEY) \
    .execute()
print("Set is_active=True in Supabase")

# ── Re-validate ──────────────────────────────────────────────────────────
r = httpx.post(
    f"{RAILWAY_URL}/v1/license/validate",
    json={
        "licenseKey": TEST_LICENSE_KEY,
        "deviceId":   TEST_DEVICE_ID,
        "platform":   "desktop"
    },
    timeout=15
)
data = r.json()

print(f"\nHTTP {r.status_code}")
print(json.dumps(data, indent=2))

if r.status_code == 403 and data.get("code") == "ERROR_CREEM_REJECTED":
    print("\n⚠️  Creem rejected — test key not in Creem sandbox. Supabase side is correct.")
else:
    assert r.status_code == 200
    assert data.get("valid") == True
    SESSION_TOKEN = data.get("token")  # Update token for heartbeat test
    print(f"\nOK ✅  Reactivated key validates — plan={data.get('plan')}")

## Step 6 — Heartbeat (Keep Session Alive)

The desktop app calls `POST /v1/license/heartbeat` every 5-10 minutes to:
1. Prove the session is still active
2. Report status (optional: account balance, equity, positions)
3. Get a refreshed token if the current one is near expiry

> **Requires:** A valid `SESSION_TOKEN` from Step 2 or 5.

In [ ]:
if not SESSION_TOKEN:
    print("⚠️  No session token — Step 2 or 5 was rejected by Creem.")
    print("    Skipping heartbeat test. This will work when using a real Creem key.")
else:
    r = httpx.post(
        f"{RAILWAY_URL}/v1/license/heartbeat",
        json={
            "token":    SESSION_TOKEN,
            "deviceId": TEST_DEVICE_ID,
            "status":   {"balance": 10000.00, "equity": 10250.00, "positions": 3}
        },
        timeout=15
    )
    data = r.json()

    print(f"HTTP {r.status_code}")
    print(json.dumps(data, indent=2))

    assert r.status_code == 200
    assert data.get("valid") == True
    if data.get("newToken"):
        SESSION_TOKEN = data["newToken"]
        print(f"Token refreshed: {SESSION_TOKEN[:20]}...")
    print(f"\nOK ✅  Heartbeat acknowledged — TTL remaining: {data.get('ttlSeconds')}s")

## Step 7 — Deactivate Device (Free License Slot)

When a user wants to move their license to a new machine, they call
`POST /v1/license/deactivate` which:
1. Marks the device as `is_active=false` in `license_devices`
2. Deletes all sessions for that device
3. Returns the number of devices still registered

> **Requires:** Step 2 to have successfully registered a device.

In [ ]:
# Check if device was registered (Step 2 succeeded past Creem)
dev_check = db.table("license_devices") \
    .select("id") \
    .eq("license_id", TEST_LICENSE_ID) \
    .eq("device_id", TEST_DEVICE_ID) \
    .eq("is_active", True) \
    .execute()

if not dev_check.data:
    print("⚠️  No active device found — Step 2 was likely rejected by Creem.")
    print("    Skipping deactivation test.")
else:
    r = httpx.post(
        f"{RAILWAY_URL}/v1/license/deactivate",
        json={
            "licenseKey": TEST_LICENSE_KEY,
            "deviceId":   TEST_DEVICE_ID
        },
        timeout=15
    )
    data = r.json()

    print(f"HTTP {r.status_code}")
    print(json.dumps(data, indent=2))

    assert r.status_code == 200
    assert data.get("success") == True
    print(f"\nOK ✅  Device deactivated — {data.get('devicesRemaining')} device(s) remaining")

## Step 8 — Cleanup

Remove all test rows from Supabase so the database is clean.
Cascade: sessions → devices → license.

In [ ]:
# ── Cascade: delete sessions and devices first ────────────────────────────
lic_row = db.table("licenses").select("id").eq("license_key", TEST_LICENSE_KEY).execute()

if lic_row.data:
    lic_id = lic_row.data[0]["id"]
    sessions = db.table("license_sessions").delete().eq("license_id", lic_id).execute()
    devices  = db.table("license_devices").delete().eq("license_id", lic_id).execute()
    print(f"Deleted {len(sessions.data)} session(s)")
    print(f"Deleted {len(devices.data)} device(s)")

# ── Delete the license row ────────────────────────────────────────────────
lic = db.table("licenses").delete().eq("license_key", TEST_LICENSE_KEY).execute()
print(f"Deleted {len(lic.data)} license row(s)")

# ── Also clean up any validation logs ─────────────────────────────────────
try:
    db.table("license_validation_logs").delete().like("license_key", "VALTEST%").execute()
    print("Cleaned up validation logs")
except Exception:
    pass  # Table may not exist in all environments

print("\nOK ✅  Cleanup complete — Supabase is back to its original state")

## Summary

| # | Test | Pass Condition |
|---|------|----------------|
| 0 | Setup + seed | Config loaded, test license inserted into Supabase |
| 1 | Health check | Railway returns `status=healthy` |
| 2 | Validate active key | HTTP 200, `valid=true`, token + plan returned |
| 3 | Validate fake key | HTTP 401 or 403, `valid=false` |
| 4 | Validate inactive key | HTTP 403, `ERROR_INACTIVE` or `ERROR_CREEM_REJECTED` |
| 5 | Reactivate + validate | HTTP 200, `valid=true` |
| 6 | Heartbeat | HTTP 200, `valid=true`, TTL remaining |
| 7 | Deactivate device | HTTP 200, `success=true`, device count decremented |
| 8 | Cleanup | All test rows removed |

---

### Integration with Creem Test Mode

The validation endpoint cross-checks with Creem **before** checking Supabase.
For full end-to-end testing:

1. **Create a test product** in the Creem sandbox dashboard
2. **Create a test license** via Creem API or dashboard — note the license key
3. Use that real Creem-generated key in Step 0 instead of the locally-seeded key
4. Railway will successfully cross-check with Creem sandbox API

When using locally-seeded keys (as this notebook does by default), Creem will
reject the key since it doesn't exist in their system. The notebook handles
this gracefully with diagnostic messages.

---

### How the Desktop App Uses This

```typescript
// In Electron app (simplified)
const response = await fetch('https://api.hedge-edge.com/v1/license/validate', {
    method: 'POST',
    headers: { 'Content-Type': 'application/json' },
    body: JSON.stringify({
        licenseKey: userInput.trim().toUpperCase(),
        deviceId:   getOrCreateDeviceId(),  // stable per-machine UUID
        platform:   'desktop'
    })
});
const data = await response.json();

if (data.valid) {
    // Store token for heartbeats
    // Enable app features based on data.features
} else {
    // Show error: data.message
}
```